### 00. tilelang导入

In [1]:
import tilelang

print(tilelang.__version__)

Loading tilelang libs from dev root: /data/caojian/workspace/git/github/tile-ai/tilelang/build
/data/caojian/uv-envs/cjuv312_tile-ai_tilelang/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/data/caojian/uv-envs/cjuv312_tile-ai_tilelang/lib/python3.12/site-packages/torch_npu/_init/core/optional_features.py:20: UserWarning: On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default. Do not set it to 1 to prevent some unknown errors
  warnings.warn(


0.1.15+cuda.git545be125


### 01. lower 纯 python 函数

In [2]:
import tilelang

try:
    def main():
        pass

    tilelang.lower(main)
except Exception as e:
    print(f"Error: {e}")

Error: Mismatched type on argument #1 when calling: `transform.RunPass(0: transform.Pass, 1: RValueRef<ir.IRModule>) -> ir.IRModule`. Expected `RValueRef<ir.IRModule>` but got `ffi.Function`


### 02. lower 被 T.prim_func 装饰的 python 函数

In [3]:
import tilelang
import tilelang.language as T

try:
    @T.prim_func
    def main():
        pass

    tilelang.lower(main)
except Exception as e:
    print(f"ERROR: {e}")

ERROR: Check failed: rewriter.NumRewrittenKernels() > 0 (0 vs. 0) : AutoSchedule could not find a tilelang_root


### 03. lower 被 T.prim_func 装饰的 python 函数 且 函数体包含 with T.Kernel()

In [4]:
import tilelang
import tilelang.language as T

@T.prim_func
def main():
    with T.Kernel(1) as _:
        pass
    pass

tilelang.lower(main)
compiled_artifact = tilelang.lower(main)
print(compiled_artifact.kernel_source)


#include <tl_templates/ascend/common.h>
#include <tl_templates/ascend/debug.h>
extern "C" __global__ __vector__ void main_kernel() {
  asc_init();
}




### 04. func 改造为 add

In [5]:
import tilelang
import tilelang.ascend.language as T

@T.prim_func
def main(
    A: T.Tensor((1,1), "float32"), # type: ignore
    B: T.Tensor((1,1), "float32"), # type: ignore
    C: T.Tensor((1,1), "float32"), # type: ignore
):
    with T.Kernel(1) as _:
        a_ub = T.alloc_shared((1,1), "float32")
        b_ub = T.alloc_shared((1,1), "float32")
        c_ub = T.alloc_shared((1,1), "float32")

        T.copy(A[0, 0], a_ub)
        T.copy(B[0, 0], b_ub)

        c_ub[0, 0] = a_ub[0, 0] + b_ub[0, 0]

        T.copy(c_ub, C[0, 0])

compiled_artifact = tilelang.lower(main)
print(compiled_artifact.kernel_source)

#include <tl_templates/ascend/common.h>
#include <tl_templates/ascend/debug.h>
extern "C" __global__ __vector__ void main_kernel(__gm__ float* A, __gm__ float* B, __gm__ float* C) {
  asc_init();
  __ubuf__ uint8_t *buf_dyn_shmem = (__ubuf__ uint8_t *)0;
  asc_copy_gm2ub_align((__ubuf__ uint8_t*)((&(((__ubuf__ float*)buf_dyn_shmem)[0]))), (__gm__ uint8_t*)((&(A[0]))), 1, 4, 0, 0, 0, static_cast<asc_load_l2_cache_mode>(0), 4, 4);
  asc_copy_gm2ub_align((__ubuf__ uint8_t*)((&(((__ubuf__ float*)buf_dyn_shmem)[8]))), (__gm__ uint8_t*)((&(B[0]))), 1, 4, 0, 0, 0, static_cast<asc_load_l2_cache_mode>(0), 4, 4);
  asc_sync_notify(PIPE_MTE2, PIPE_S, static_cast<event_t>(0));
  asc_sync_wait(PIPE_MTE2, PIPE_S, static_cast<event_t>(0));
  ((__ubuf__ float*)buf_dyn_shmem)[16] = (((__ubuf__ float*)buf_dyn_shmem)[0] + ((__ubuf__ float*)buf_dyn_shmem)[8]);
  asc_sync_notify(PIPE_S, PIPE_MTE3, static_cast<event_t>(0));
  asc_sync_wait(PIPE_S, PIPE_MTE3, static_cast<event_t>(0));
  asc_copy_ub2gm_align(